In [96]:
# Imports and data loading

import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

df_trans = pd.read_csv(r"C:\Users\user\Desktop\AI-Cohort\csv\fraud_transactions.csv")
df_cust = pd.read_csv(r"C:\Users\user\Desktop\AI-Cohort\csv\customer_profiles.csv")

df_trans["timestamp"] = pd.to_datetime(df_trans["timestamp"])

print("Transactions:", df_trans.shape)
print("Customers:", df_cust.shape)

Transactions: (12000, 8)
Customers: (1000, 5)


In [97]:

# UNDERSTAND THE CUSTOMER DATA


print("Customer columns:")
print(df_cust.columns.tolist())

print("\nFirst 5 customers:")
print(df_cust.head())

print("\nMissing values:")
print(df_cust.isnull().sum())


# UNDERSTAND THE TRANSACTION DATA


print("Transaction columns:")
print(df_trans.columns.tolist())

print("\nFirst 5 Transaction:")
print(df_trans.head())

print("\nMissing values:")
print(df_trans.isnull().sum())

Customer columns:
['transaction_id', 'customer_id', 'amount', 'timestamp', 'location', 'merchant', 'merchant_category', 'fraud']

First 5 customers:
  transaction_id customer_id   amount           timestamp   location  \
0         T00301       C0031  6527.78 2026-08-01 08:02:00       Pune   
1         T06641       C0665  2408.40 2026-08-01 08:02:00        NaN   
2         T04211       C0422  5004.81 2026-08-01 08:03:00  Bengaluru   
3         T01061       C0107   765.39 2026-08-01 08:03:00     Mumbai   
4         T04541       C0455  1520.69 2026-08-01 08:04:00  Bengaluru   

      merchant merchant_category  fraud  
0  Burger King        Restaurant      0  
1         Spar           Grocery      0  
2          H&M           Apparel      0  
3          Max           Apparel      0  
4  Burger King        Restaurant      0  

Missing values:
transaction_id         0
customer_id            0
amount                 0
timestamp              0
location             546
merchant             386

In [98]:
# FRAUD DISTRIBUTION


print("Fraud counts:")
print(df_trans["fraud"].value_counts())

print("\nFraud percentages:")
print(df_trans["fraud"].value_counts(normalize=True).mul(100).round(2))

Fraud counts:
fraud
0    10000
1     2000
Name: count, dtype: int64

Fraud percentages:
fraud
0    83.33
1    16.67
Name: proportion, dtype: float64


In [99]:
# BASIC TRANSACTION STATISTICS


print("Transaction amount statistics:")
print(
    df_trans["amount"].describe()
)

print("\nNumber of unique customers:")
print(
    df_trans["customer_id"].nunique()
)

print("\nNumber of unique merchants:")
print(
    df_trans["merchant"].nunique()
)

print("\nNumber of unique locations:")
print(
    df_trans["location"].nunique()
)

Transaction amount statistics:
count    12000.000000
mean      3543.863585
std       3944.802417
min        271.300000
25%       1563.365000
50%       2723.940000
75%       4186.612500
max      43237.630000
Name: amount, dtype: float64

Number of unique customers:
1000

Number of unique merchants:
10

Number of unique locations:
8


In [100]:
#  TRANSACTION AMOUNT VS FRAUD

print(df_trans.groupby("fraud")["amount"].describe())

print("\nAverage transaction amount by fraud status:")

print(df_trans.groupby("fraud")["amount"].mean())

         count         mean          std     min        25%       50%  \
fraud                                                                   
0      10000.0  2772.847672  1580.455707  271.30  1473.5250  2547.145   
1       2000.0  7398.943150  7941.702136  423.78  2350.7875  4159.655   

            75%       max  
fraud                      
0      3828.925   7380.49  
1      9405.605  43237.63  

Average transaction amount by fraud status:
fraud
0    2772.847672
1    7398.943150
Name: amount, dtype: float64


In [101]:
# Add customer profile information

df_trans = df_trans.merge(df_cust[["customer_id", "home_location"]],on="customer_id",how="left")

print(df_trans[["customer_id", "location", "home_location"]].head())

print("Customers without profile match:",df_trans["home_location"].isnull().sum())

  customer_id   location home_location
0       C0031       Pune          Pune
1       C0665        NaN     Hyderabad
2       C0422  Bengaluru     Bengaluru
3       C0107     Mumbai        Mumbai
4       C0455  Bengaluru     Bengaluru
Customers without profile match: 0


In [102]:
#SORT TRANSACTIONS CHRONOLOGICALLY

df_trans = df_trans.sort_values(["customer_id","timestamp"]).reset_index(drop=True)

print(df_trans[["transaction_id","customer_id","timestamp","amount"]].head(10))

  transaction_id customer_id           timestamp  amount
0         T00001       C0001 2026-08-01 15:19:00  799.20
1         T00002       C0001 2026-08-02 08:00:00  401.66
2         T00003       C0001 2026-08-03 15:34:00  516.92
3         T00004       C0001 2026-08-04 15:19:00  886.33
4         T00005       C0001 2026-08-05 17:14:00  795.07
5         T00006       C0001 2026-08-06 22:35:00  421.78
6         T00007       C0001 2026-08-07 19:41:00  406.29
7         T00008       C0001 2026-08-08 09:27:00  441.40
8         T00009       C0001 2026-08-09 19:34:00  480.65
9         T00010       C0001 2026-08-10 09:11:00  421.43


In [103]:
# CREATE CUSTOMER HISTORIES

customer_histories = {customer_id: customer_df.copy() for customer_id, customer_df in df_trans.groupby("customer_id")}

print("Number of customers:",len(customer_histories))

Number of customers: 1000


# Evidence used by the agent -

1. Amount unusual: Transaction amount is much higher than the customer's historical spending pattern.
2. Location unusual: Transaction occurs in a location where the customer has not previously transacted.
3. Merchant unusual: Transaction occurs at a merchant where the customer has not previously transacted.
4. Frequency unusual: The time gap between the current transaction and the previous transaction is much shorter than the customer's normal transaction interval.
5. Velocity unusual: The customer has made multiple transactions within a short time window, indicating unusually concentrated transaction activity.

In [104]:
#  GET CUSTOMER HISTORY

def get_customer_history(customer_id,timestamp,customer_histories):
    """
    Return only transactions that happened
    before the current transaction.
    """

    customer = customer_histories[customer_id]

    current_time = pd.to_datetime(timestamp)

    history = customer[customer["timestamp"] < current_time]

    return history

In [105]:
# TEST CUSTOMER HISTORY
test_transaction = df_trans.iloc[100]

history = get_customer_history(test_transaction["customer_id"],test_transaction["timestamp"],customer_histories)

print("Current transaction:")
print(test_transaction[["transaction_id","customer_id","timestamp"]])

print("\nPrevious transactions:")
print(history[["transaction_id","timestamp","amount"]].tail())

Current transaction:
transaction_id                 T00088
customer_id                     C0009
timestamp         2026-08-08 15:02:00
Name: 100, dtype: object

Previous transactions:
   transaction_id           timestamp   amount
95         T00083 2026-08-03 11:35:00  5621.84
96         T00084 2026-08-04 12:49:00  3621.77
97         T00085 2026-08-05 14:27:00  2988.55
98         T00086 2026-08-06 17:41:00  4004.40
99         T00087 2026-08-07 13:39:00  3265.31


In [106]:
# AMOUNT UNUSUAL

def check_amount(current_amount,history):
    """
    Returns True when the current transaction
    is more than 4x the customer's historical
    average transaction amount.
    """

    if history.empty:
        return False

    average_amount = history["amount"].mean()

    if average_amount == 0:
        return False

    return (current_amount >4 * average_amount)

In [107]:
# LOCATION UNUSUAL

def check_location(current_location,historical_locations):
    """
    Returns:
    True  -> location has not been seen before
    False -> location has been seen before
    None  -> location is missing
    """

    if pd.isna(current_location):
        return None

    if current_location in historical_locations.values:
        return False

    return True

In [108]:
#  MERCHANT UNUSUAL

def check_merchant(current_merchant,historical_merchants):
    """
    Returns:
    True  -> merchant has not been seen before
    False -> merchant has been seen before
    None  -> merchant is missing
    """

    if pd.isna(current_merchant):
        return None

    if current_merchant in historical_merchants.values:
        return False

    return True

In [109]:
# FREQUENCY UNUSUAL

def check_frequency(current_timestamp,history):
    """
    Compare the current transaction gap
    with the customer's historical median gap.

    A transaction is unusual when the current gap
    is less than 50% of the normal median gap.
    """

    if history.empty:
        return False

    current_timestamp = pd.to_datetime(current_timestamp)

    history = history.sort_values("timestamp")

    previous_transaction = history.iloc[-1]["timestamp"]

    current_gap = (current_timestamp -previous_transaction)

    historical_gaps = (history["timestamp"].diff().dropna())

    if historical_gaps.empty:
        return False

    normal_median_gap = (historical_gaps.median())

    frequency_threshold = (normal_median_gap * 0.5)

    return (current_gap <frequency_threshold)

In [110]:
#  VELOCITY UNUSUAL

def check_velocity(current_timestamp,history):
    """
    Returns True when at least 3 previous
    transactions occurred within 30 minutes.
    """

    if history.empty:
        return False

    current_timestamp = pd.to_datetime(current_timestamp)

    time_difference_minutes = ((current_timestamp -history["timestamp"]).dt.total_seconds()/ 60)

    recent_transactions = history[(time_difference_minutes >= 0) & (time_difference_minutes <= 30)]

    return (len(recent_transactions) >= 3)

In [111]:
# 23. GENERATE ALL EVIDENCE

def get_evidence(transaction,customer_histories):

    history = get_customer_history(transaction["customer_id"],transaction["timestamp"],customer_histories)

    amount_unusual = check_amount(transaction["amount"],history)

    location_unusual = check_location(transaction["location"],history["location"])
    
    merchant_unusual = check_merchant(transaction["merchant"],history["merchant"])

    frequency_unusual = check_frequency(transaction["timestamp"],history)

    velocity_unusual = check_velocity(transaction["timestamp"],history)

    return {
        "amount_unusual":
            amount_unusual,

        "location_unusual":
            location_unusual,

        "merchant_unusual":
            merchant_unusual,

        "frequency_unusual":
            frequency_unusual,

        "velocity_unusual":
            velocity_unusual
    }

In [112]:
# GENERATE EVIDENCE FOR ALL TRANSACTIONS

evidence_rows = []

for _, transaction in df_trans.iterrows():

    evidence = get_evidence(
        transaction,
        customer_histories
    )

    evidence_rows.append(evidence)

evidence_df = pd.DataFrame(
    evidence_rows
)

print("Evidence dataframe shape:", evidence_df.shape)

print("\nEvidence columns:")
print(evidence_df.columns.tolist())

print("\nFirst 5 rows:")
print(evidence_df.head())

Evidence dataframe shape: (12000, 5)

Evidence columns:
['amount_unusual', 'location_unusual', 'merchant_unusual', 'frequency_unusual', 'velocity_unusual']

First 5 rows:
   amount_unusual location_unusual merchant_unusual  frequency_unusual  \
0           False             True             True              False   
1           False            False             True              False   
2           False            False            False              False   
3           False            False            False              False   
4           False            False            False              False   

   velocity_unusual  
0             False  
1             False  
2             False  
3             False  
4             False  


In [113]:
#  ADD EVIDENCE TO TRANSACTION DATASET

evidence_columns = [
    "amount_unusual",
    "location_unusual",
    "merchant_unusual",
    "frequency_unusual",
    "velocity_unusual"
]

for column in evidence_columns:

    df_trans[column] = evidence_df[column].values

print(
    df_trans[
        [
            "transaction_id",
            "amount",
            "location",
            "merchant",
            "amount_unusual",
            "location_unusual",
            "merchant_unusual",
            "frequency_unusual",
            "velocity_unusual",
            "fraud"
        ]
    ].head(10)
)

  transaction_id  amount location   merchant  amount_unusual location_unusual  \
0         T00001  799.20   Mumbai     Myntra           False             True   
1         T00002  401.66   Mumbai        H&M           False            False   
2         T00003  516.92   Mumbai     Myntra           False            False   
3         T00004  886.33   Mumbai     Myntra           False            False   
4         T00005  795.07   Mumbai     Myntra           False            False   
5         T00006  421.78   Mumbai        H&M           False            False   
6         T00007  406.29   Mumbai     Myntra           False            False   
7         T00008  441.40   Mumbai     Myntra           False            False   
8         T00009  480.65   Mumbai  Big Chill           False            False   
9         T00010  421.43   Mumbai        H&M           False            False   

  merchant_unusual  frequency_unusual  velocity_unusual  fraud  
0             True              False      

In [114]:
# EVIDENCE DISTRIBUTION
for column in evidence_columns:

    print("\n" + "=" * 60)
    print(column.upper())
    print("=" * 60)

    print(
        df_trans[column]
        .value_counts(dropna=False)
    )
    
#  EVIDENCE PERCENTAGES BY FRAUD CLASS

for column in evidence_columns:

    print("\n" + "=" * 60)
    print(column.upper())
    print("=" * 60)

    result = pd.crosstab(
        df_trans[column],
        df_trans["fraud"],
        normalize="columns"
    ) * 100

    print(
        result.round(2)
    )
# COUNT SUSPICIOUS SIGNALS

df_trans["suspicious_signal_count"] = (df_trans[evidence_columns].fillna(False).astype(int).sum(axis=1))

print(
    df_trans[
        "suspicious_signal_count"
    ].value_counts()
    .sort_index()
)


AMOUNT_UNUSUAL
amount_unusual
False    11471
True       529
Name: count, dtype: int64

LOCATION_UNUSUAL
location_unusual
False    9743
True     1711
None      546
Name: count, dtype: int64

MERCHANT_UNUSUAL
merchant_unusual
False    7826
True     3788
None      386
Name: count, dtype: int64

FREQUENCY_UNUSUAL
frequency_unusual
False    9715
True     2285
Name: count, dtype: int64

VELOCITY_UNUSUAL
velocity_unusual
False    12000
Name: count, dtype: int64

AMOUNT_UNUSUAL
fraud               0      1
amount_unusual              
False           100.0  73.55
True              0.0  26.45

LOCATION_UNUSUAL
fraud                 0      1
location_unusual              
False             89.52  62.89
True              10.48  37.11

MERCHANT_UNUSUAL
fraud                 0      1
merchant_unusual              
False             65.06  79.03
True              34.94  20.97

FREQUENCY_UNUSUAL
fraud                  0      1
frequency_unusual              
False              97.15    0.0
True     

C:\Users\user\AppData\Local\Temp\ipykernel_19192\566637950.py:32: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_trans["suspicious_signal_count"] = (df_trans[evidence_columns].fillna(False).astype(int).sum(axis=1))


In [116]:
#  SUSPICIOUS SIGNAL COUNT VS FRAUD
print(pd.crosstab(df_trans["suspicious_signal_count"],df_trans["fraud"],normalize="columns").round(3))

fraud                        0      1
suspicious_signal_count              
0                        0.636  0.000
1                        0.262  0.293
2                        0.102  0.602
3                        0.000  0.096
4                        0.000  0.010


In [117]:
# AVERAGE SUSPICIOUS SIGNALS

print(df_trans.groupby("fraud")["suspicious_signal_count"].mean())

fraud
0    0.4668
1    1.8225
Name: suspicious_signal_count, dtype: float64


In [118]:
# INSPECT TRANSACTIONS WITH MANY SIGNALS

suspicious_transactions = df_trans[df_trans["suspicious_signal_count"] >= 2]

print(suspicious_transactions[
        [
            "transaction_id",
            "customer_id",
            "amount",
            "location",
            "merchant",
            "amount_unusual",
            "location_unusual",
            "merchant_unusual",
            "frequency_unusual",
            "velocity_unusual",
            "suspicious_signal_count",
            "fraud"
        ]
    ].head(20)
)

   transaction_id customer_id    amount   location   merchant  amount_unusual  \
0          T00001       C0001    799.20     Mumbai     Myntra           False   
10      AMT_C0001       C0001   3132.72     Mumbai        H&M            True   
11      LOC_C0001       C0001    609.03  Hyderabad        H&M           False   
12         T00011       C0002   4336.25        Goa   Reliance           False   
22    MERCH_C0002       C0002  13047.24      Delhi  Big Chill           False   
23         T00021       C0003    449.59    Chennai        H&M           False   
33    MERCH_C0003       C0003   1986.01  Bengaluru       Spar           False   
34      LOC_C0003       C0003    736.32    Kolkata     Myntra           False   
35         T00031       C0004   1693.66      Delhi   Flipkart           False   
45      AMT_C0004       C0004  18236.25      Delhi  Big Chill            True   
46         T00041       C0005   3217.40       Pune        Max           False   
56      LOC_C0005       C000

In [119]:
# FREQUENCY ANALYSIS

print(pd.crosstab(df_trans["frequency_unusual"],df_trans["fraud"]))

print("\nPercentage within each fraud class:")

print(pd.crosstab(df_trans["frequency_unusual"],df_trans["fraud"],normalize="columns").round(3))

fraud                 0     1
frequency_unusual            
False              9715     0
True                285  2000

Percentage within each fraud class:
fraud                  0    1
frequency_unusual            
False              0.972  0.0
True               0.028  1.0


In [120]:
#  VELOCITY ANALYSIS

print( pd.crosstab( df_trans["velocity_unusual"],df_trans["fraud"]))

print("\nPercentage within each fraud class:")

print( pd.crosstab(df_trans["velocity_unusual"], df_trans["fraud"],normalize="columns").round(3))

fraud                 0     1
velocity_unusual             
False             10000  2000

Percentage within each fraud class:
fraud               0    1
velocity_unusual          
False             1.0  1.0


In [121]:
# LOCATION ANALYSIS

print(pd.crosstab(df_trans["location_unusual"],df_trans["fraud"]))

print("\nPercentage within each fraud class:")

print( pd.crosstab(df_trans["location_unusual"],df_trans["fraud"],normalize="columns").round(3))

fraud                0     1
location_unusual            
False             8538  1205
True              1000   711

Percentage within each fraud class:
fraud                 0      1
location_unusual              
False             0.895  0.629
True              0.105  0.371


In [122]:
# MERCHANT ANALYSIS
print(
    pd.crosstab(
        df_trans["merchant_unusual"],
        df_trans["fraud"]
    )
)

print("\nPercentage within each fraud class:")

print(
    pd.crosstab(
        df_trans["merchant_unusual"],
        df_trans["fraud"],
        normalize="columns"
    ).round(3)
)

fraud                0     1
merchant_unusual            
False             6300  1526
True              3383   405

Percentage within each fraud class:
fraud                 0     1
merchant_unusual             
False             0.651  0.79
True              0.349  0.21


In [123]:
# AMOUNT ANALYSIS
print(
    pd.crosstab(
        df_trans["amount_unusual"],
        df_trans["fraud"]
    )
)

print("\nPercentage within each fraud class:")

print(
    pd.crosstab(
        df_trans["amount_unusual"],
        df_trans["fraud"],
        normalize="columns"
    ).round(3)
)

fraud               0     1
amount_unusual             
False           10000  1471
True                0   529

Percentage within each fraud class:
fraud             0      1
amount_unusual            
False           1.0  0.736
True            0.0  0.264


In [124]:
# ============================================================
# FINAL WEEK 1 DATASET
# ============================================================

week1_columns = [
    "transaction_id",
    "customer_id",
    "amount",
    "timestamp",
    "location",
    "merchant",
    "merchant_category",
    "home_location",

    # Week 1 observable evidence
    "amount_unusual",
    "location_unusual",
    "merchant_unusual",
    "frequency_unusual",
    "velocity_unusual",

    # Exploratory feature
    "suspicious_signal_count",

    # Actual label - used only for evaluation
    "fraud"
]

week1_df = df_trans[
    week1_columns
].copy()

print(
    "Week 1 dataset shape:",
    week1_df.shape
)

print(
    "\nWeek 1 columns:"
)

print(
    week1_df.columns.tolist()
)

print(
    "\nFirst 10 rows:"
)

print(
    week1_df.head(10)
)

Week 1 dataset shape: (12000, 15)

Week 1 columns:
['transaction_id', 'customer_id', 'amount', 'timestamp', 'location', 'merchant', 'merchant_category', 'home_location', 'amount_unusual', 'location_unusual', 'merchant_unusual', 'frequency_unusual', 'velocity_unusual', 'suspicious_signal_count', 'fraud']

First 10 rows:
  transaction_id customer_id  amount           timestamp location   merchant  \
0         T00001       C0001  799.20 2026-08-01 15:19:00   Mumbai     Myntra   
1         T00002       C0001  401.66 2026-08-02 08:00:00   Mumbai        H&M   
2         T00003       C0001  516.92 2026-08-03 15:34:00   Mumbai     Myntra   
3         T00004       C0001  886.33 2026-08-04 15:19:00   Mumbai     Myntra   
4         T00005       C0001  795.07 2026-08-05 17:14:00   Mumbai     Myntra   
5         T00006       C0001  421.78 2026-08-06 22:35:00   Mumbai        H&M   
6         T00007       C0001  406.29 2026-08-07 19:41:00   Mumbai     Myntra   
7         T00008       C0001  441.40 20

In [125]:
# Calculate likelihoods
########## Among fraudlent transactions, how often X is unusual
def calculate_likelihoods(df_trans):
    fraud_trans = df_trans.loc[df_trans["fraud"] == 1]
    legit_trans = df_trans.loc[df_trans["fraud"] == 0]

    total_fraud = len(fraud_trans)
    total_legit = len(legit_trans)

    fraud_amount_count = (fraud_trans["amount_unusual"] == True).sum()
    legit_amount_count = (legit_trans["amount_unusual"] == True).sum()

    fraud_location_count = (fraud_trans["location_unusual"] == True).sum()
    legit_location_count = (legit_trans["location_unusual"] == True).sum()

    fraud_merchant_count = (fraud_trans["merchant_unusual"] == True).sum()
    legit_merchant_count = (legit_trans["merchant_unusual"] == True).sum()

    fraud_frequency_count = (fraud_trans["frequency_unusual"] == True).sum()
    legit_frequency_count = (legit_trans["frequency_unusual"] == True).sum()

    fraud_velocity_count = (fraud_trans["velocity_unusual"] == True).sum()
    legit_velocity_count = (legit_trans["velocity_unusual"] == True).sum()

    # Laplace smoothing
    p_amount_fraud = (fraud_amount_count + 1) / (total_fraud + 2)
    p_amount_legit = (legit_amount_count + 1) / (total_legit + 2)

    p_location_fraud = (fraud_location_count + 1) / (total_fraud + 2)
    p_location_legit = (legit_location_count + 1) / (total_legit + 2)

    p_merchant_fraud = (fraud_merchant_count + 1) / (total_fraud + 2)
    p_merchant_legit = (legit_merchant_count + 1) / (total_legit + 2)

    p_frequency_fraud = (fraud_frequency_count + 1) / (total_fraud + 2)
    p_frequency_legit = (legit_frequency_count + 1) / (total_legit + 2)

    p_velocity_fraud = (fraud_velocity_count + 1) / (total_fraud + 2)
    p_velocity_legit = (legit_velocity_count + 1) / (total_legit + 2)

    return (
        p_amount_fraud, p_amount_legit,
        p_location_fraud, p_location_legit,
        p_merchant_fraud, p_merchant_legit,
        p_frequency_fraud, p_frequency_legit,
        p_velocity_fraud, p_velocity_legit
    )


likelihoods = calculate_likelihoods(df_trans)
print(likelihoods)

(np.float64(0.2647352647352647), np.float64(9.998000399920016e-05), np.float64(0.35564435564435565), np.float64(0.10007998400319935), np.float64(0.20279720279720279), np.float64(0.33833233353329334), np.float64(0.9995004995004995), np.float64(0.028594281143771244), np.float64(0.0004995004995004995), np.float64(9.998000399920016e-05))


In [126]:
#  Calculate fraud belief

def calculate_fraud_belief(evidence, likelihoods):

    prior_fraud = 0.05
    prior_legit = 1 - prior_fraud

    (
        p_amount_fraud,
        p_amount_legit,
        p_location_fraud,
        p_location_legit,
        p_merchant_fraud,
        p_merchant_legit,
        p_frequency_fraud,
        p_frequency_legit,
        p_velocity_fraud,
        p_velocity_legit
    ) = likelihoods

    if evidence["amount_unusual"] == True:
        amount_fraud = p_amount_fraud  #If unusual amount is observed, use P(unusual amount | fraud).
        amount_legit = p_amount_legit  #If normal amount is observed, use P(normal amount | fraud).
    else:
        amount_fraud = 1 - p_amount_fraud
        amount_legit = 1 - p_amount_legit

    if evidence["location_unusual"] == True:
        location_fraud = p_location_fraud
        location_legit = p_location_legit
    elif evidence["location_unusual"] == False:
        location_fraud = 1 - p_location_fraud
        location_legit = 1 - p_location_legit
    else:
        location_fraud = 1
        location_legit = 1

    if evidence["merchant_unusual"] == True:
        merchant_fraud = p_merchant_fraud
        merchant_legit = p_merchant_legit
    elif evidence["merchant_unusual"] == False:
        merchant_fraud = 1 - p_merchant_fraud
        merchant_legit = 1 - p_merchant_legit
    else:
        merchant_fraud = 1
        merchant_legit = 1

    if evidence["frequency_unusual"] == True:
        frequency_fraud = p_frequency_fraud
        frequency_legit = p_frequency_legit
    else:
        frequency_fraud = 1 - p_frequency_fraud
        frequency_legit = 1 - p_frequency_legit

    if evidence["velocity_unusual"] == True:
        velocity_fraud = p_velocity_fraud
        velocity_legit = p_velocity_legit
    else:
        velocity_fraud = 1 - p_velocity_fraud
        velocity_legit = 1 - p_velocity_legit

    fraud_probability = (prior_fraud * amount_fraud * location_fraud * merchant_fraud * frequency_fraud * velocity_fraud)

    legit_probability = (prior_legit  * amount_legit * location_legit * merchant_legit * frequency_legit * velocity_legit)

    return fraud_probability / (fraud_probability + legit_probability)

In [127]:
#  Cost model and action selection

def calculate_action_costs(fraud_belief):
    legit_belief = 1 - fraud_belief

    approve_cost = fraud_belief * 100
    question_cost = (fraud_belief * 10 + legit_belief * 2)
    examine_cost = (fraud_belief * 5  + legit_belief * 8)
    decline_cost = legit_belief * 20

    return {
        "Approve": approve_cost,
        "Question": question_cost,
        "Examine": examine_cost,
        "Decline": decline_cost
    }


def choose_action(costs):
    return min(costs, key=costs.get)

In [128]:
def make_decision(evidence,likelihoods):
    fraud_belief = calculate_fraud_belief(evidence, likelihoods)
    costs = calculate_action_costs(fraud_belief)
    action = choose_action(costs)
    
    return (fraud_belief,costs,action)

In [129]:
test_transaction = df_trans.iloc[2000]
print(test_transaction)

transaction_id                          T01683
customer_id                              C0169
amount                                 4941.59
timestamp                  2026-08-03 08:04:00
location                               Kolkata
merchant                                Myntra
merchant_category                   E-commerce
fraud                                        0
home_location                          Kolkata
amount_unusual                           False
location_unusual                         False
merchant_unusual                          True
frequency_unusual                        False
velocity_unusual                         False
suspicious_signal_count                      1
Name: 2000, dtype: object


In [130]:
result = make_decision(test_transaction, likelihoods)
print(result)

(np.float64(8.537520162478538e-06), {'Approve': np.float64(0.0008537520162478538), 'Question': np.float64(2.0000683001613), 'Examine': np.float64(7.999974387439512), 'Decline': np.float64(19.99982924959675)}, 'Approve')


In [131]:
results = []

for _,trans in df_trans.iterrows():
    evidence = {
    "amount_unusual": trans["amount_unusual"],
    "location_unusual": trans["location_unusual"],
    "merchant_unusual": trans["merchant_unusual"],
    "frequency_unusual": trans["frequency_unusual"],
    "velocity_unusual": trans["velocity_unusual"]}
    
    fraud_belief, costs, action = make_decision(evidence, likelihoods)
    results.append({
    "transaction_id":trans["transaction_id"],
    "fraud_belief": fraud_belief,
    "action":action,
    "actual_fraud":trans["fraud"]})
results_df = pd.DataFrame(results)
print(results_df["action"].value_counts())

action
Approve     9715
Decline     1213
Question    1021
Examine       51
Name: count, dtype: int64


In [132]:
print(results_df.head())

  transaction_id  fraud_belief   action  actual_fraud
0         T00001      0.000042  Approve             0
1         T00002      0.000009  Approve             0
2         T00003      0.000017  Approve             0
3         T00004      0.000017  Approve             0
4         T00005      0.000017  Approve             0


In [133]:
print(pd.crosstab(results_df["action"],results_df["actual_fraud"]))

actual_fraud     0     1
action                  
Approve       9715     0
Decline          0  1213
Examine          9    42
Question       276   745


In [135]:
results_df["predicted_fraud"] = (results_df["action"] == "Decline").astype(int)
print(results_df[["action", "predicted_fraud", "actual_fraud"]].head(20))

     action  predicted_fraud  actual_fraud
0   Approve                0             0
1   Approve                0             0
2   Approve                0             0
3   Approve                0             0
4   Approve                0             0
5   Approve                0             0
6   Approve                0             0
7   Approve                0             0
8   Approve                0             0
9   Approve                0             0
10  Decline                1             1
11  Decline                1             1
12  Approve                0             0
13  Approve                0             0
14  Approve                0             0
15  Approve                0             0
16  Approve                0             0
17  Approve                0             0
18  Approve                0             0
19  Approve                0             0


In [136]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
accuracy = accuracy_score(
    results_df["actual_fraud"],
    results_df["predicted_fraud"]
)

print("Accuracy:", accuracy)

Accuracy: 0.9344166666666667


In [137]:
precision = precision_score(
    results_df["actual_fraud"],
    results_df["predicted_fraud"]
)

print("Precision:", precision)

Precision: 1.0


In [138]:
recall = recall_score(
    results_df["actual_fraud"],
    results_df["predicted_fraud"]
)

print("Recall:", recall)


Recall: 0.6065


In [139]:
f1 = f1_score(
    results_df["actual_fraud"],
    results_df["predicted_fraud"]
)

print("F1:", f1)


F1: 0.7550575785869903


In [140]:
print(results_df["fraud_belief"].describe())

count    12000.000000
mean         0.135591
std          0.295286
min          0.000009
25%          0.000017
50%          0.000017
75%          0.000042
max          0.999952
Name: fraud_belief, dtype: float64


In [141]:
print(results_df.groupby("actual_fraud")["fraud_belief"].mean())

actual_fraud
0    0.014291
1    0.742092
Name: fraud_belief, dtype: float64


In [142]:
print(results_df.groupby("action")["fraud_belief"].describe())

           count      mean       std       min       25%       50%       75%  \
action                                                                         
Approve   9715.0  0.000018  0.000010  0.000009  0.000014  0.000017  0.000017   
Decline   1213.0  0.899949  0.095024  0.742294  0.852720  0.852720  0.999762   
Examine     51.0  0.617910  0.008780  0.574879  0.619666  0.619666  0.619666   
Question  1021.0  0.493406  0.073163  0.367236  0.447683  0.538443  0.538443   

               max  
action              
Approve   0.000085  
Decline   0.999952  
Examine   0.619666  
Question  0.538443  


In [143]:
# beliefs are not smoothly distributed. They are concentrated in a few distinct regions.
bins = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

print(
    pd.cut(
        results_df["fraud_belief"],
        bins=bins,
        include_lowest=True
    ).value_counts().sort_index()
)

fraud_belief
(-0.001, 0.1]    9715
(0.1, 0.2]          0
(0.2, 0.3]          0
(0.3, 0.4]        249
(0.4, 0.5]         54
(0.5, 0.6]        720
(0.6, 0.7]         49
(0.7, 0.8]        180
(0.8, 0.9]        504
(0.9, 1.0]        529
Name: count, dtype: int64


# Conclusion
The Week 1 agent uses five behavioral signals to estimate fraud belief and selects among four cost-sensitive actions: Approve, Question, Examine, and Decline. When only Decline is treated as a positive fraud prediction, the agent achieves 93.44% accuracy, 100% precision, 60.65% recall, and 75.51% F1 on the evaluated dataset. Many fraudulent transactions are routed to Question or Examine rather than directly Declined, suggesting that additional information could help resolve uncertain cases.